# Phase 5 — Analyse comparative

Les trois portefeuilles tiennent les mêmes titres, sur les mêmes mois, avec les mêmes rendements en CHF et le même SARON. Le seul changement d'une colonne à l'autre est la règle de poids.

Ce notebook lit les CSV déjà produits. Il ne recalcule pas les poids. Les formules sont dans `documentation/05_portfolio_construction.ipynb`. Le détail arithmétique est dans `reports/portfolio/05_calculs_detailles.ipynb`.

Les chiffres des sections suivantes sont imprimés depuis les CSV de cette exécution. Les poids de la volatilité minimale et du Sharpe maximal sont réestimés sur toute la fenêtre à chaque arrivée de données. Un mois de plus change aussi les poids des mois déjà publiés. La fenêtre s'arrête au dernier mois du SARON.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src.config import MARKET_HISTORY_YEARS, REPORTING_CURRENCY

from src.portfolio_construction.build_portfolios import (
    COMPARISON_FILE,
    METRICS_FILE,
    MONTHLY_RETURNS_FILE,
    WEIGHTS_FILE,
)

metrics = pd.read_csv(METRICS_FILE)
comparison = pd.read_csv(COMPARISON_FILE)
monthly = pd.read_csv(MONTHLY_RETURNS_FILE)
weights = pd.read_csv(WEIGHTS_FILE)

labels = {
    "equal_weight": "Équipondéré",
    "minimum_volatility": "Volatilité minimale",
    "maximum_sharpe": "Sharpe maximal",
}
table = metrics.set_index("strategy").loc[list(labels)]

def fr(value, digits):
    return f"{value:.{digits}f}".replace(".", ",")


def fr_pct(value, digits=1):
    return fr(float(value) * 100, digits) + " %"


def final_wealth(strategy):
    series = monthly.loc[monthly["strategy"].eq(strategy), "portfolio_return"]
    return float((1 + series).prod())


def above_count(strategy, threshold=0.01):
    group = weights.loc[weights["strategy"].eq(strategy)]
    return int(group["weight"].ge(threshold).sum())


def top_holding(strategy):
    group = weights.loc[weights["strategy"].eq(strategy)].nlargest(1, "weight")
    row = group.iloc[0]
    return str(row["name"]), float(row["weight"])


def following_names(strategy, count=4):
    group = weights.loc[weights["strategy"].eq(strategy)].nlargest(count + 1, "weight")
    return ", ".join(group["name"].astype(str).iloc[1:].tolist())


def describe_equal_weight():
    row = table.loc["equal_weight"]
    n = int(row["names"])
    print(
        "\n".join(
            [
                f"Chaque titre pèse 1/{n}, soit {fr_pct(1 / n)}.",
                f"Le nombre effectif de titres est {fr(row['effective_holdings'], 2)}.",
                (
                    f"Sur {row['window_start']} – {row['window_end']}, "
                    f"soit {int(row['months'])} mois, ce portefeuille rend "
                    f"{fr_pct(row['annualized_return'])} par an."
                ),
                f"Sa volatilité annualisée est {fr_pct(row['annualized_volatility'])}.",
                f"Son Sharpe, après SARON, est {fr(row['sharpe_ratio'], 2)}.",
                (
                    f"Un franc investi au départ vaut {fr(final_wealth('equal_weight'), 2)} "
                    "franc à la fin de la fenêtre."
                ),
                f"La pire perte depuis un sommet est de {fr_pct(-row['maximum_drawdown'])}.",
                "Ces chiffres sont la référence de cette exécution.",
            ]
        )
    )


def describe_minimum_volatility():
    base = table.loc["equal_weight"]
    row = table.loc["minimum_volatility"]
    name, weight = top_holding("minimum_volatility")
    gap = (float(base["annualized_volatility"]) - float(row["annualized_volatility"])) * 100
    if gap >= 0:
        gap_text = f"{fr(gap, 1)} points de pourcentage sous l'équipondéré"
    else:
        gap_text = f"{fr(-gap, 1)} points de pourcentage au-dessus de l'équipondéré"
    print(
        "\n".join(
            [
                f"La volatilité annualisée est {fr_pct(row['annualized_volatility'])}, soit {gap_text}.",
                f"Le nombre effectif de titres est {fr(row['effective_holdings'], 1)}.",
                f"{above_count('minimum_volatility')} titres dépassent 1 %.",
                f"Le plus gros poids est {name}, à {fr_pct(weight)}.",
                f"Les poids suivants sont {following_names('minimum_volatility')}.",
                (
                    f"Le rendement annualisé est {fr_pct(row['annualized_return'])}. "
                    f"Le Sharpe est {fr(row['sharpe_ratio'], 2)}, "
                    f"contre {fr(base['sharpe_ratio'], 2)} pour l'équipondéré."
                ),
                (
                    f"La baisse maximale est {fr_pct(-row['maximum_drawdown'])}, "
                    f"contre {fr_pct(-base['maximum_drawdown'])} pour l'équipondéré."
                ),
                f"La richesse finale est {fr(final_wealth('minimum_volatility'), 2)}.",
            ]
        )
    )


def describe_maximum_sharpe():
    base = table.loc["equal_weight"]
    low = table.loc["minimum_volatility"]
    row = table.loc["maximum_sharpe"]
    name, weight = top_holding("maximum_sharpe")
    print(
        "\n".join(
            [
                (
                    f"Sur {row['window_start']} – {row['window_end']}, ce rapport vaut "
                    f"{fr(row['sharpe_ratio'], 2)}, contre {fr(base['sharpe_ratio'], 2)} "
                    f"pour l'équipondéré et {fr(low['sharpe_ratio'], 2)} pour la volatilité minimale."
                ),
                (
                    f"Le rendement annualisé est {fr_pct(row['annualized_return'])}. "
                    f"La volatilité annualisée est {fr_pct(row['annualized_volatility'])}, "
                    f"contre {fr_pct(base['annualized_volatility'])} pour l'équipondéré."
                ),
                (
                    f"La baisse maximale est {fr_pct(-row['maximum_drawdown'])}, "
                    f"contre {fr_pct(-base['maximum_drawdown'])} pour l'équipondéré."
                ),
                f"Le nombre effectif de titres est {fr(row['effective_holdings'], 1)}.",
                f"{above_count('maximum_sharpe')} titres dépassent 1 %.",
                f"Le plus gros poids est {name}, à {fr_pct(weight)}, puis {following_names('maximum_sharpe')}.",
                f"Un franc investi au départ vaut {fr(final_wealth('maximum_sharpe'), 2)} francs à la fin de la fenêtre.",
                (
                    "Ces poids ont été choisis avec les rendements des mêmes mois. "
                    "Le Sharpe publié dit que ce mélange a été le meilleur mélange long only "
                    "de la fenêtre, une fois la fenêtre connue."
                ),
            ]
        )
    )


def describe_wealth():
    row = table.loc["equal_weight"]
    print(
        f"L'indice part de 1 avant {row['window_start']}. "
        f"Les fins d'année sont gardées, ainsi que {row['window_end']}, dernier mois de la fenêtre."
    )
    for strategy, label in labels.items():
        print(f"Richesse finale, {label} : {fr(final_wealth(strategy), 2)}.")
    print("Cet écart de richesse décrit les poids estimés sur cette fenêtre.")


def describe_overview():
    base = table.loc["equal_weight"]
    low = table.loc["minimum_volatility"]
    high = table.loc["maximum_sharpe"]
    print(
        "\n".join(
            [
                (
                    f"L'équipondéré couvre les {int(base['names'])} titres. "
                    f"Son Sharpe est {fr(base['sharpe_ratio'], 2)} et sa volatilité "
                    f"{fr_pct(base['annualized_volatility'])}."
                ),
                (
                    f"La volatilité minimale a {fr(low['effective_holdings'], 1)} titres effectifs "
                    f"et une volatilité de {fr_pct(low['annualized_volatility'])}. "
                    f"Son Sharpe est {fr(low['sharpe_ratio'], 2)}."
                ),
                (
                    f"Le Sharpe maximal atteint {fr(high['sharpe_ratio'], 2)}, "
                    f"avec une volatilité de {fr_pct(high['annualized_volatility'])} "
                    f"et une richesse finale de {fr(final_wealth('maximum_sharpe'), 2)}."
                ),
                (
                    "Comme les poids optimisés sont estimés sur les mois qui servent ensuite "
                    "à calculer les ratios, ces chiffres sont ceux de l'échantillon."
                ),
            ]
        )
    )


## 1. Les trois résultats

Le rendement annualisé est le taux géométrique des mois de la fenêtre. La volatilité est celle des rendements du portefeuille. Le Sharpe retire le SARON mensuel avant de diviser par le risque. La baisse maximale est négative : elle mesure la pire perte depuis un sommet. Le nombre effectif de titres vaut le nombre de lignes lorsque tous les poids sont égaux, et il baisse lorsque les poids se concentrent.


In [2]:
shown = table.copy()
shown.index = shown.index.map(labels)
for column in ("annualized_return", "annualized_volatility", "maximum_drawdown", "maximum_weight"):
    shown[column] = (shown[column] * 100).map(lambda value: f"{value:.1f} %")
shown["sharpe_ratio"] = shown["sharpe_ratio"].map(lambda value: f"{value:.2f}")
shown["effective_holdings"] = shown["effective_holdings"].map(lambda value: f"{value:.1f}")
shown.loc[:, [
    "names",
    "months",
    "window_start",
    "window_end",
    "annualized_return",
    "annualized_volatility",
    "sharpe_ratio",
    "maximum_drawdown",
    "maximum_weight",
    "effective_holdings",
]].rename(columns={
    "names": "Titres",
    "months": "Mois",
    "window_start": "Début",
    "window_end": "Fin",
    "annualized_return": "Rendement annualisé",
    "annualized_volatility": "Volatilité annualisée",
    "sharpe_ratio": "Sharpe",
    "maximum_drawdown": "Baisse maximale",
    "maximum_weight": "Poids maximum",
    "effective_holdings": "Titres effectifs",
})


,Titres,Mois,Début,Fin,Rendement annualisé,Volatilité annualisée,Sharpe,Baisse maximale,Poids maximum,Titres effectifs
strategy,,,,,,,,,,
Équipondéré,83,120,2016-10,2026-09,9.6 %,12.6 %,0.80,-16.8 %,1.2 %,83.0
Volatilité minimale,83,120,2016-10,2026-09,3.7 %,7.6 %,0.53,-16.5 %,14.9 %,14.1
Sharpe maximal,83,120,2016-10,2026-09,32.9 %,16.0 %,1.89,-12.8 %,19.7 %,9.6


## 2. Diversification simple

L'équipondéré répond à une question précise : que se passe-t-il si aucun titre n'est préféré ? Ses poids ne lisent pas les rendements passés.


In [3]:
describe_equal_weight()


Chaque titre pèse 1/83, soit 1,2 %.
Le nombre effectif de titres est 83,00.
Sur 2016-10 – 2026-09, soit 120 mois, ce portefeuille rend 9,6 % par an.
Sa volatilité annualisée est 12,6 %.
Son Sharpe, après SARON, est 0,80.
Un franc investi au départ vaut 2,51 franc à la fin de la fenêtre.
La pire perte depuis un sommet est de 16,8 %.
Ces chiffres sont la référence de cette exécution.


## 3. Réduction du risque

La volatilité minimale cherche la plus petite variance mensuelle. Les poids viennent de la covariance des rendements de la fenêtre.


In [4]:
describe_minimum_volatility()


La volatilité annualisée est 7,6 %, soit 5,0 points de pourcentage sous l'équipondéré.
Le nombre effectif de titres est 14,1.
22 titres dépassent 1 %.
Le plus gros poids est SWISS PRIME SITE AG, à 14,9 %.
Les poids suivants sont NIPPON BUILDING FUND REIT, SAPUTO, UNICHARM, HENAN SHUANGHUI INVESTMENT & DEVEL.
Le rendement annualisé est 3,7 %. Le Sharpe est 0,53, contre 0,80 pour l'équipondéré.
La baisse maximale est 16,5 %, contre 16,8 % pour l'équipondéré.
La richesse finale est 1,44.


In [5]:
group = weights.loc[weights["strategy"].eq("minimum_volatility")].nlargest(8, "weight")
group.assign(weight=(group["weight"] * 100).map(lambda value: f"{value:.1f} %")).loc[
    :, ["yahoo_symbol", "name", "listing_currency", "weight"]
].rename(columns={
    "yahoo_symbol": "Symbole",
    "name": "Titre",
    "listing_currency": "Devise",
    "weight": "Poids",
})


,Symbole,Titre,Devise,Poids
83,SPSN.SW,SWISS PRIME SITE AG,CHF,14.9 %
84,8951.T,NIPPON BUILDING FUND REIT,JPY,11.0 %
85,SAP.TO,SAPUTO,CAD,8.8 %
86,8113.T,UNICHARM,JPY,8.4 %
87,000895.SZ,HENAN SHUANGHUI INVESTMENT & DEVEL,CNY,8.4 %
88,REC.MI,RECORDATI INDUSTRIA CHIMICA E FARM,EUR,6.1 %
89,3898.HK,ZHUZHOU CRRC TIMES ELECTRIC LTD H,HKD,4.6 %
90,NOVN.SW,NOVARTIS AG,CHF,3.6 %


## 4. Rendement ajusté du risque

Le Sharpe maximal cherche le plus grand rapport entre l'excédent au-dessus du SARON et l'écart-type de cet excédent. L'équipondéré est le seul des trois dont les poids ne dépendent pas de ces rendements.


In [6]:
describe_maximum_sharpe()


Sur 2016-10 – 2026-09, ce rapport vaut 1,89, contre 0,80 pour l'équipondéré et 0,53 pour la volatilité minimale.
Le rendement annualisé est 32,9 %. La volatilité annualisée est 16,0 %, contre 12,6 % pour l'équipondéré.
La baisse maximale est 12,8 %, contre 16,8 % pour l'équipondéré.
Le nombre effectif de titres est 9,6.
17 titres dépassent 1 %.
Le plus gros poids est NVIDIA, à 19,7 %, puis ELIA GROUP SA, VERTEX PHARMACEUTICALS, ARGENX, NOVARTIS AG.
Un franc investi au départ vaut 17,24 francs à la fin de la fenêtre.
Ces poids ont été choisis avec les rendements des mêmes mois. Le Sharpe publié dit que ce mélange a été le meilleur mélange long only de la fenêtre, une fois la fenêtre connue.


In [7]:
group = weights.loc[weights["strategy"].eq("maximum_sharpe")].nlargest(8, "weight")
group.assign(weight=(group["weight"] * 100).map(lambda value: f"{value:.1f} %")).loc[
    :, ["yahoo_symbol", "name", "listing_currency", "weight"]
].rename(columns={
    "yahoo_symbol": "Symbole",
    "name": "Titre",
    "listing_currency": "Devise",
    "weight": "Poids",
})


,Symbole,Titre,Devise,Poids
166,NVDA,NVIDIA,USD,19.7 %
167,ELI.BR,ELIA GROUP SA,EUR,12.6 %
168,VRTX,VERTEX PHARMACEUTICALS,USD,11.9 %
169,ARGX.BR,ARGENX,EUR,11.5 %
170,NOVN.SW,NOVARTIS AG,CHF,10.4 %
171,300274.SZ,SUNGROW POWER SUPPLY LTD A,CNY,6.5 %
172,UTHR,UNITED THERAPEUTICS,USD,4.1 %
173,VER.VI,VERBUND AG,EUR,3.6 %


## 5. Richesse cumulée

Chaque ligne est la valeur à la fin du mois, après capitalisation des rendements mensuels.


In [8]:
describe_wealth()


L'indice part de 1 avant 2016-10. Les fins d'année sont gardées, ainsi que 2026-09, dernier mois de la fenêtre.
Richesse finale, Équipondéré : 2,51.
Richesse finale, Volatilité minimale : 1,44.
Richesse finale, Sharpe maximal : 17,24.
Cet écart de richesse décrit les poids estimés sur cette fenêtre.


In [9]:
wealth = monthly.loc[:, ["month", "strategy", "portfolio_return"]].copy()
wealth["wealth"] = wealth.groupby("strategy")["portfolio_return"].transform(
    lambda series: (1 + series).cumprod()
)
marks = wealth["month"].str.endswith("-12") | wealth["month"].eq(wealth["month"].max())
wealth_table = wealth.loc[marks].pivot(index="month", columns="strategy", values="wealth")
wealth_table = wealth_table.loc[:, list(labels)]
wealth_table.columns = [labels[column] for column in wealth_table.columns]
wealth_table.round(2)


,Équipondéré,Volatilité minimale,Sharpe maximal
month,,,
2016-12,0.98,0.97,1.11
2017-12,1.29,1.14,1.97
2018-12,1.23,1.13,2.03
2019-12,1.54,1.33,2.91
2020-12,1.83,1.38,4.69
2021-12,2.13,1.43,6.15
2022-12,1.94,1.32,6.06
2023-12,1.92,1.25,7.80
2024-12,2.12,1.34,11.05


## 6. Lecture d'ensemble

Les trois objectifs ne désignent pas le même portefeuille.


In [10]:
describe_overview()


L'équipondéré couvre les 83 titres. Son Sharpe est 0,80 et sa volatilité 12,6 %.
La volatilité minimale a 14,1 titres effectifs et une volatilité de 7,6 %. Son Sharpe est 0,53.
Le Sharpe maximal atteint 1,89, avec une volatilité de 16,0 % et une richesse finale de 17,24.
Comme les poids optimisés sont estimés sur les mois qui servent ensuite à calculer les ratios, ces chiffres sont ceux de l'échantillon.
